In [ ]:
from embedding_atlas.widget import EmbeddingAtlasWidget
from embedding_atlas.projection import async_compute_projection
import pandas as pd
from datasets import load_dataset

In [ ]:
# 加载数据集
ds = load_dataset("james-burton/wine_reviews", split="validation")
df = pd.DataFrame(ds)

In [ ]:
# 使用默认的 Sentence Transformers 计算文本 embedding 及其投影
# 在 Jupyter notebooks 中使用 async_compute_projection（因为其中已有正在运行的 event loop）
df = await async_compute_projection(df, inputs="description", modality="text", x="projection_x", y="projection_y", neighbors="neighbors")

In [ ]:
# 使用 Embedding Atlas widget 显示数据集
w = EmbeddingAtlasWidget(df, text="description", x="projection_x", y="projection_y", neighbors="neighbors")
w

In [ ]:
# 从 widget 获取当前选择，并返回为 dataframe
w.selection()

Embedding Atlas 支持使用 [LiteLLM](https://docs.litellm.ai/docs/embedding/supported_embedding) 支持的所有模型来运行文本 embeddings。

要运行下面的示例，请先安装 [Ollama](https://ollama.com/download)，并运行以下命令下载 [nomic-embed-text](https://ollama.com/library/nomic-embed-text) 模型：

```bash
ollama pull nomic-embed-text
```

In [ ]:
# 使用本地运行的 Ollama API 计算文本 embedding 及其投影
# 对本地服务器，建议将 max_concurrency 设为较小值（例如 2-4），
# 以免服务器过载。设为 1 则按顺序处理。
df = await async_compute_projection(
    df,
    inputs="description",
    modality="text",
    x="projection_x",
    y="projection_y",
    neighbors="neighbors",
    embedder="litellm",
    embedder_args={"api_base": "http://localhost:11434"},
    model="ollama/nomic-embed-text",
    batch_size=512,
    max_concurrency=2,
)

In [ ]:
# 使用 Embedding Atlas widget 显示数据集，embeddings 来自 Ollama API 提供的 nomic-embed-text
EmbeddingAtlasWidget(df, text="description", x="projection_x", y="projection_y", neighbors="neighbors")

In [ ]:
# 使用 OpenAI API 计算文本 embedding 及其投影
df = await async_compute_projection(
    df,
    inputs="description",
    modality="text",
    x="projection_x",
    y="projection_y",
    neighbors="neighbors",
    embedder="litellm",
    # 你的 OpenAI API key。也可以省略此项，改为设置 OPENAI_API_KEY 环境变量。
    embedder_args={"api_key": "sk-xxx"},
    model="openai/text-embedding-3-small",
    # OpenAI 每次请求的输入上限为 300K tokens，因此应根据平均 `text` 条目长度选择 batch size
    batch_size=1024,
)

In [ ]:
# 使用 Embedding Atlas widget 显示数据集，embeddings 来自 OpenAI API 提供的 text-embedding-3-small
EmbeddingAtlasWidget(df, text="description", x="projection_x", y="projection_y", neighbors="neighbors")